# 🔧 Feature Engineering — Complete Guide
---
Covers: Missing Values, Encoding, Outliers, Scaling, Feature Selection, New Features

In [ ]:
import pandas as pd, numpy as np, warnings
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')

df = pd.read_csv('loan_default_dataset.csv')
print(f'Original shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum()>0]}')

## 1. Missing Values
- **Mean** → Numerical, no outliers
- **Median** → Numerical, HAS outliers (robust!)
- **Mode** → Categorical
- **KNN Imputer** → Uses neighbors to guess

In [ ]:
df_clean = df.copy()
for c in df_clean.select_dtypes(include=[np.number]).columns:
    if df_clean[c].isnull().sum() > 0:
        df_clean[c].fillna(df_clean[c].median(), inplace=True)  # WHY median? Robust to outliers!
        print(f'  Filled {c} with median')
for c in df_clean.select_dtypes(include=['object']).columns:
    if df_clean[c].isnull().sum() > 0:
        df_clean[c].fillna(df_clean[c].mode()[0], inplace=True)
        print(f'  Filled {c} with mode')
print(f'\n✅ Missing after: {df_clean.isnull().sum().sum()}')

## 2. Outliers (IQR Method)
```
IQR = Q3 - Q1
Lower = Q1 - 1.5*IQR
Upper = Q3 + 1.5*IQR
```

In [ ]:
num_cols = [c for c in df_clean.select_dtypes(include=[np.number]).columns if c != 'Default']
for c in num_cols:
    Q1, Q3 = df_clean[c].quantile(0.25), df_clean[c].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df_clean[c] < Q1-1.5*IQR) | (df_clean[c] > Q3+1.5*IQR)).sum()
    if outliers > 0:
        df_clean[c] = df_clean[c].clip(Q1-1.5*IQR, Q3+1.5*IQR)  # CAP, don't remove!
        print(f'  {c}: {outliers} outliers capped')
print('✅ Outliers handled!')

## 3. Encoding
- **Label** → Ordinal (Low/Med/High)
- **One-Hot** → Nominal (Color, City)

In [ ]:
for c in df_clean.select_dtypes(include=['object']).columns:
    if df_clean[c].nunique() == 2:
        df_clean[c] = LabelEncoder().fit_transform(df_clean[c])
        print(f'  {c}: Label Encoded')
    else:
        dummies = pd.get_dummies(df_clean[c], prefix=c, drop_first=True)
        df_clean = pd.concat([df_clean.drop(c, axis=1), dummies], axis=1)
        print(f'  {c}: One-Hot Encoded ({dummies.shape[1]} cols)')
print(f'\n✅ Shape after encoding: {df_clean.shape}')

## 4. Scaling: Standard vs MinMax

In [ ]:
X = df_clean.drop('Default', axis=1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
col = num_cols[0] if num_cols else X.columns[0]

axes[0].hist(X[col], bins=30, color='steelblue', edgecolor='black')
axes[0].set_title(f'Original: {col}')

ss = StandardScaler()
axes[1].hist(ss.fit_transform(X[[col]]), bins=30, color='coral', edgecolor='black')
axes[1].set_title('StandardScaler (mean=0, std=1)')

mm = MinMaxScaler()
axes[2].hist(mm.fit_transform(X[[col]]), bins=30, color='green', edgecolor='black')
axes[2].set_title('MinMaxScaler ([0, 1])')

plt.tight_layout()
plt.show()
print('💡 StandardScaler: robust to outliers | MinMaxScaler: bounded [0,1]')

## 5. Feature Selection

In [ ]:
y = df_clean['Default']
X_scaled = StandardScaler().fit_transform(X)

# SelectKBest
k = min(10, X.shape[1])
selector = SelectKBest(f_classif, k=k)
selector.fit(X_scaled, y)
top_features = X.columns[selector.get_support()].tolist()
print(f'Top {k} features (SelectKBest): {top_features}')

# Correlation
corr = X.corrwith(y).abs().sort_values(ascending=False)
print(f'\nTop correlated with target:\n{corr.head(10)}')

## 6. Creating New Features

In [ ]:
if 'Income' in df_clean.columns and 'LoanAmount' in df_clean.columns:
    df_clean['Debt_to_Income'] = df_clean['LoanAmount'] / (df_clean['Income'] + 1)
    print('✅ Created: Debt_to_Income')
if 'Income' in df_clean.columns:
    df_clean['Log_Income'] = np.log1p(df_clean['Income'])
    print('✅ Created: Log_Income')
if 'Age' in df_clean.columns:
    df_clean['Age_Group'] = pd.cut(df_clean['Age'], bins=[0,25,35,50,100],
                                    labels=[0,1,2,3]).astype(float).fillna(0).astype(int)
    print('✅ Created: Age_Group')
print(f'\nFinal shape: {df_clean.shape}')